# Graph-aware PaySim fraud detection: a reproducible walkthrough

**Audience:** ML engineers and fraud analysts familiar with Python and precision–recall curves. This notebook uses committed aggregate results, so it runs without the 471 MB PaySim CSV. To reproduce training, follow the README.

**You will see:** the chronological split, the paired graph ablation, the 1% validation-FPR operating point, and an optional local drift snapshot.


## 1. Load the published run summary

The source file is generated by `scripts/export_results.py` after a local training run. No outcome value is typed into this notebook.


In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
summary = json.loads((ROOT / "docs/results.json").read_text())
print("Source:", summary["dataset"], "| sampled fraction:", summary["sample_fraction"])
print("Step boundaries:", summary["split"])
pd.DataFrame(summary["counts"]).T


## 2. Compare the prespecified models

Model selection uses validation PR-AUC. The final test window is later in time and is read only after that choice. Recall uses a threshold chosen to allow at most 1% false positives among validation negatives; the realized test FPR may differ.


In [ ]:
rows = []
for name, result in summary["models"].items():
    rows.append({"model": name,
                 "validation PR-AUC": result["validation"]["pr_auc"],
                 "test PR-AUC": result["test"]["pr_auc"],
                 "test recall @ validation 1% FPR": result["test"]["recall_at_validation_1pct_fpr"],
                 "realized test FPR": result["test"]["realized_fpr"]})
comparison = pd.DataFrame(rows).set_index("model")
comparison.round(4)


## 3. Interpret graph lift

A graph result is useful only if it improves the same held-out transactions over the transaction-only XGBoost control. PaySim has almost one unique sender per row; several sender-side graph features are therefore usually zero.


In [ ]:
lift = summary["graph_pr_auc_lift_test"]
print(f"Graph minus transaction-only test PR-AUC: {lift:+.4f}")
print("Serving choice (validation-selected):", summary["selected_on_validation"])


## 4. Inspect the latest local drift check (optional)

`artifacts/` is intentionally absent from Git. If you trained locally, run `fraud-lab monitor` and optionally `fraud-lab monitor --amount-multiplier 3` first.


In [ ]:
status_path = ROOT / "artifacts/monitor_status.json"
if status_path.exists():
    status = json.loads(status_path.read_text())
    print(status["status"], "| amount KS", round(status["amount_ks"]["statistic"], 3),
          "| score KS", round(status["score_ks"]["statistic"], 3))
else:
    print("No local monitor snapshot. The committed model comparison above is still complete.")


## Exercise and answer scaffold

**Question:** If graph PR-AUC is lower on test but higher on validation, which model is allowed to serve under the precommitted rule? Why should the test outcome still be reported?

**Answer scaffold:** The serving choice is `selected_on_validation`; the test set estimates its later-time behavior. Report both graph and baseline test metrics to expose the graph hypothesis, but never use this test observation to tune another model and call it untouched.

**Extension:** Replace uniform sampling with a larger fraction and rerun the same frozen protocol. Do not compare scores from different test cohorts as if they were paired.
